# Simsalabim — End-to-End Training on Google Colab

This notebook runs the **entire model pipeline** for Simsalabim from scratch on Colab:

1. Mount Google Drive (audio + manifest live there).
2. Clone the repo and install the `model/` requirements.
3. Inspect the dataset (`manifest.csv`).
4. Generate **song-level splits** (train / val / test).
5. Compute **per-bin mel statistics** over the train split (one-shot).
6. Build `TakesDataset` + `DataLoader` for train and eval.
7. Visualize one preprocessed sample (waveform, log-mel, f0 contour).
8. Build the **two-stream encoder** (ResNet-18 + 1D-CNN) and **Sub-center ArcFace** loss.
9. Sanity-train a few epochs.
10. Run the **full training loop** with periodic retrieval evaluation.
11. Plot loss / mAP curves.
12. Build a **FAISS gallery** from val songs.
13. Report **top-1 / top-5 / MRR / mAP@10** broken down by style.
14. **Inference** on a single WAV.
15. Save artifacts (checkpoint, stats, splits, gallery) back to Drive.

Read `model/PLAN.md` first — every choice in this notebook (10 s window, log-mel 80 bins, sub-center ArcFace, mAP@10, etc.) is justified there.

---

**Before you start:**

- Set the runtime to **GPU** (`Runtime → Change runtime type → T4 or better`).
- In your Google Drive, place the dataset at:
  ```
  My Drive/SimlabimAI/dataset/data/manifest.csv
  My Drive/SimlabimAI/dataset/data/raw_audio/<song-slug>/<take-uuid>.wav
  ```
  i.e. mirror the layout of the local `dataset/data/` folder.
- You can either clone the repo from GitHub (default below) or upload the `model/` and `shared/` folders into `My Drive/SimlabimAI/` manually.

## 1. Runtime check

Confirm we have a GPU. Without one the training loop will work but will be ~30× slower.

In [ ]:
import subprocess, sys
print('python:', sys.version.split()[0])
try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode().strip())
except Exception as err:
    print('no GPU detected:', err)

## 2. Mount Google Drive

We expect the dataset (manifest + audio) under `My Drive/SimlabimAI/dataset/data/`. Adjust `DRIVE_ROOT` below if your layout differs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/SimlabimAI')
DATASET_ROOT = DRIVE_ROOT / 'dataset' / 'data'      # contains manifest.csv + raw_audio/
ARTIFACTS_ROOT = DRIVE_ROOT / 'model_artifacts'     # outputs (checkpoints, gallery, etc.)
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)

MANIFEST_CSV = DATASET_ROOT / 'manifest.csv'
assert MANIFEST_CSV.exists(), f'missing {MANIFEST_CSV} — upload your dataset to Drive first'
print('dataset OK at', DATASET_ROOT)

## 3. Clone the repo and install dependencies

We clone the repo into `/content/SimlabimAI` so the local source tree and the Drive dataset stay independent. If you've forked it, edit the URL.

The `model/` package validates `shared/wav.json`, `shared/slugs.json`, etc. at import time, so the `shared/` folder must be present beside `model/`.

In [ ]:
REPO_URL = 'https://github.com/Gustavoo-Pacheco/SimlabimAI.git'
REPO_DIR = Path('/content/SimlabimAI')

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only

MODEL_DIR = REPO_DIR / 'model'
assert MODEL_DIR.exists() and (REPO_DIR / 'shared').exists(), 'repo layout mismatch'
print('repo OK at', REPO_DIR)

In [ ]:
# Install the model/ dependencies. Colab already ships with torch + numpy, so we only
# pip-install the missing pieces — keeps the install under ~2 min on a fresh runtime.
!pip install -q audiomentations pesto-pitch pyloudnorm pytorch-metric-learning faiss-cpu fast_mp3_augment soundfile pyyaml tqdm

In [ ]:
import sys
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

import yaml
from src.io import load_wav, EXPECTED_SR
from src.preproc import SAMPLE_RATE, crop_or_pad, normalize_loudness, trim_by_confidence
from src.representation import (
    MelConfig, MelExtractor, PestoConfig, PestoExtractor,
    coarse_confidence_for_trim, normalize_log_mel,
)
from src.augment_waveform import build_waveform_augmenter
from src.augment_mel import SpecAugmenter, SpecAugmentConfig
from src.dataset import load_manifest, build_song_id_map, TakesDataset
from src.splits import make_splits, save_splits, load_splits, is_gallery_take
from src.encoder import TwoStreamEncoder, EncoderConfig
from src.loss import SongArcFaceLoss, ArcFaceConfig
from src.train import (
    make_loader, train_one_epoch, evaluate_retrieval,
    save_checkpoint, load_checkpoint, TrainState,
)
from src.enroll import build_gallery, Gallery
from src.infer import infer_wav

print('SR =', EXPECTED_SR, '— all imports OK')

## 4. Load the configs

The three YAMLs in `model/configs/` are the single source of truth for hyperparameters. We read them once here and pass them down — never hardcode values in the notebook.

In [ ]:
PREPROC = yaml.safe_load((MODEL_DIR / 'configs/preproc.yaml').read_text())
MODEL_CFG = yaml.safe_load((MODEL_DIR / 'configs/model.yaml').read_text())
TRAIN_CFG = yaml.safe_load((MODEL_DIR / 'configs/train.yaml').read_text())
INFER_CFG = yaml.safe_load((MODEL_DIR / 'configs/infer.yaml').read_text())

import json
print(json.dumps({'preproc': PREPROC, 'model': MODEL_CFG, 'train': TRAIN_CFG, 'infer': INFER_CFG}, indent=2))

## 5. Inspect the manifest

We expect one row per take with at least: `take_id, song_slug, style, status, storage_key`. We accept `approved` and `pending` here so the toy dataset is usable for sanity training — drop `pending` once you have enough approved takes.

In [ ]:
import csv
from collections import Counter

ACCEPTED_STATUSES = ('approved', 'pending')

with MANIFEST_CSV.open() as f:
    rows = [r for r in csv.DictReader(f) if r['status'] in ACCEPTED_STATUSES]

songs = Counter(r['song_slug'] for r in rows)
styles = Counter(r['style'] for r in rows)
print(f'takes: {len(rows)}')
print(f'songs: {len(songs)} unique')
print(f'styles: {dict(styles)}')
print('takes per song (top 10):', songs.most_common(10))

## 6. Make song-level splits

**Split unit = song**, not take. This is critical: the encoder must generalize to *unseen songs*, not memorize specific takes. See PLAN.md [6.1].

We persist splits to Drive so they're reusable across notebook restarts.

In [ ]:
SPLITS_PATH = ARTIFACTS_ROOT / 'splits.json'
SPLIT_SEED = 0
RATIOS = (0.7, 0.15, 0.15)

if SPLITS_PATH.exists():
    splits = load_splits(SPLITS_PATH)
    print(f'loaded existing splits from {SPLITS_PATH}')
else:
    slugs = [r['song_slug'] for r in rows]
    splits = make_splits(slugs, ratios=RATIOS, seed=SPLIT_SEED)
    save_splits(splits, SPLITS_PATH)
    print(f'wrote new splits to {SPLITS_PATH}')

print(f'train: {len(splits.train)} songs  ({splits.train[:5]}{"..." if len(splits.train) > 5 else ""})')
print(f'val:   {len(splits.val)} songs   ({splits.val})')
print(f'test:  {len(splits.test)} songs  ({splits.test})')

## 7. Build the manifest rows per split

We materialize three lists of `ManifestRow` — one per split. The `audio_root` is `DATASET_ROOT` because `storage_key` is `raw_audio/<song>/<take>.wav` (per `shared/storage.json`).

In [ ]:
all_manifest = load_manifest(MANIFEST_CSV, DATASET_ROOT, statuses=ACCEPTED_STATUSES)

train_rows = [r for r in all_manifest if r.song_slug in set(splits.train)]
val_rows   = [r for r in all_manifest if r.song_slug in set(splits.val)]
test_rows  = [r for r in all_manifest if r.song_slug in set(splits.test)]

song_id_map = build_song_id_map(train_rows)         # encoder learns IDs for train songs
val_id_map  = build_song_id_map(val_rows)           # val/test are evaluated by retrieval — they need their own ID space
test_id_map = build_song_id_map(test_rows)

print('train rows:', len(train_rows), '— classes:', len(song_id_map))
print('val rows:  ', len(val_rows),   '— val songs:', len(val_id_map))
print('test rows: ', len(test_rows),  '— test songs:', len(test_id_map))

## 8. Compute mel statistics (per-bin mean/std)

Log-mel needs **per-bin standardization** so the CNN sees zero-mean / unit-std inputs. Stats are computed over the **train split only**, no augmentation, center crop. Saved to Drive — only re-run when the splits or preproc change.

This runs PESTO + LUFS per take, so it scales linearly with `len(train_rows)`. Expect ~2–4 s per take on a T4.

In [ ]:
import json, torch
from tqdm.auto import tqdm

STATS_PATH = ARTIFACTS_ROOT / 'stats.json'

def compute_mel_stats(rows, preproc_cfg):
    mel_cfg = MelConfig(**{k: preproc_cfg['mel'][k] for k in (
        'n_fft', 'win_length', 'hop_length', 'n_mels', 'f_min', 'f_max', 'mel_scale', 'power'
    )})
    mel_extractor = MelExtractor(mel_cfg)
    duration_samples = int(preproc_cfg['crop']['duration_s'] * SAMPLE_RATE)
    trim = preproc_cfg['trim']

    n_mels = mel_cfg.n_mels
    sum_x  = torch.zeros(n_mels, dtype=torch.float64)
    sum_x2 = torch.zeros(n_mels, dtype=torch.float64)
    count = 0

    for r in tqdm(rows, desc='mel stats'):
        wav = load_wav(r.audio_path)
        conf = coarse_confidence_for_trim(wav, step_size_ms=trim['step_size_ms'])
        wav = trim_by_confidence(wav, conf,
            frame_hop_ms=trim['step_size_ms'],
            threshold=trim['confidence_threshold'],
            margin_ms=trim['margin_ms'])
        wav = normalize_loudness(wav, target_lufs=preproc_cfg['loudness']['target_lufs'])
        wav = crop_or_pad(wav, duration_samples, mode='center')
        mel = mel_extractor(wav).to(torch.float64)
        sum_x  += mel.sum(dim=1)
        sum_x2 += (mel * mel).sum(dim=1)
        count += mel.shape[1]

    mean = (sum_x / count)
    var = (sum_x2 / count) - mean**2
    std = torch.sqrt(var.clamp_min(0.0))
    return {'n_mels': n_mels, 'mean': mean.tolist(), 'std': std.tolist(),
            'frames_aggregated': count, 'takes': len(rows)}

if STATS_PATH.exists():
    print(f'loaded existing stats from {STATS_PATH}')
else:
    stats = compute_mel_stats(train_rows, PREPROC)
    STATS_PATH.write_text(json.dumps(stats, indent=2))
    print(f'wrote {STATS_PATH}')

stats = json.loads(STATS_PATH.read_text())
print(f'n_mels={stats["n_mels"]}  takes={stats["takes"]}  frames={stats["frames_aggregated"]}')

## 9. Build datasets and DataLoaders

The train dataset gets the waveform + spec augmenters; val/test get neither. Stats are loaded from Drive.

In [ ]:
mel_cfg = MelConfig(**{k: PREPROC['mel'][k] for k in (
    'n_fft', 'win_length', 'hop_length', 'n_mels', 'f_min', 'f_max', 'mel_scale', 'power'
)})
pesto_cfg = PestoConfig(
    step_size_ms=PREPROC['pesto']['step_size_ms'],
    confidence_threshold=PREPROC['pesto']['confidence_threshold'],
    median_subtract=PREPROC['pesto']['median_subtract'],
)

waveform_aug = build_waveform_augmenter(PREPROC['augment']['waveform'])
sa = PREPROC['augment']['mel']['spec_augment']
spec_aug = SpecAugmenter(SpecAugmentConfig(
    time_mask_p=sa['time_mask']['p'], time_mask_count=sa['time_mask']['count'], time_mask_max=sa['time_mask']['max_width'],
    freq_mask_p=sa['freq_mask']['p'], freq_mask_count=sa['freq_mask']['count'], freq_mask_max=sa['freq_mask']['max_width'],
))

common = dict(
    mel_cfg=mel_cfg, pesto_cfg=pesto_cfg,
    mel_stats_path=STATS_PATH,
    crop_duration_s=PREPROC['crop']['duration_s'],
    trim_step_size_ms=PREPROC['trim']['step_size_ms'],
    trim_confidence_threshold=PREPROC['trim']['confidence_threshold'],
    trim_margin_ms=PREPROC['trim']['margin_ms'],
    target_lufs=PREPROC['loudness']['target_lufs'],
)

train_ds = TakesDataset(train_rows, song_id_map, train=True,
                        waveform_augmenter=waveform_aug, spec_augmenter=spec_aug, **common)
val_ds   = TakesDataset(val_rows,   val_id_map,   train=False, **common)
test_ds  = TakesDataset(test_rows,  test_id_map,  train=False, **common)

print(f'len(train)={len(train_ds)}, len(val)={len(val_ds)}, len(test)={len(test_ds)}')

## 10. Visualize one sample

Sanity check: a single take through the *eval* pipeline (no augmentation). We plot waveform, log-mel, and the f0 contour. If the mel looks black or the f0 is flat zero, the preprocessing is broken — fix it before training.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sample = val_ds[0] if len(val_ds) else train_ds[0]
mel = sample['mel'].numpy()        # [n_mels, T]
f0  = sample['f0'].numpy()         # [2, T_f0]

raw = load_wav(val_rows[0].audio_path if val_rows else train_rows[0].audio_path).numpy()

fig, axes = plt.subplots(3, 1, figsize=(12, 7))
axes[0].plot(np.arange(len(raw)) / SAMPLE_RATE, raw, lw=0.4)
axes[0].set_title(f'raw waveform — {sample["take_id"]} ({sample["style"]})')
axes[0].set_xlabel('s'); axes[0].set_ylabel('amp')

axes[1].imshow(mel, origin='lower', aspect='auto', cmap='magma')
axes[1].set_title(f'log-mel (normalized)  shape={mel.shape}')
axes[1].set_ylabel('mel bin')

axes[2].plot(f0[0], label='pitch (median-subtracted semitones)')
axes[2].plot(f0[1], label='confidence', alpha=0.7)
axes[2].set_title(f'f0 stream  shape={f0.shape}')
axes[2].set_xlabel('frame'); axes[2].legend()
plt.tight_layout(); plt.show()

## 11. Build the encoder, loss, optimizer, scheduler

- **Encoder**: two-stream, mel through ResNet-18 (1-channel stem), f0 through a 1D-CNN, fusion → L2-normalized 256-d embedding.
- **Loss**: Sub-center ArcFace over `len(song_id_map)` train classes. Its prototypes are learnable parameters and **go into the optimizer**.
- **Optimizer**: AdamW with cosine schedule + warmup.

In [ ]:
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR, LambdaLR

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)

encoder = TwoStreamEncoder(EncoderConfig(
    embedding_dim=MODEL_CFG['encoder']['embedding_dim'],
    fusion_hidden=MODEL_CFG['encoder']['fusion_hidden'],
    dropout=MODEL_CFG['encoder']['dropout'],
    f0_in_channels=MODEL_CFG['encoder']['f0_in_channels'],
)).to(device)

loss_fn = SongArcFaceLoss(ArcFaceConfig(
    num_classes=len(song_id_map),
    embedding_size=MODEL_CFG['encoder']['embedding_dim'],
    margin=MODEL_CFG['arcface']['margin'],
    scale=MODEL_CFG['arcface']['scale'],
    sub_centers=MODEL_CFG['arcface']['sub_centers'],
)).to(device)

params = list(encoder.parameters()) + list(loss_fn.parameters())
optimizer = AdamW(params, lr=TRAIN_CFG['optimizer']['lr'],
                  weight_decay=TRAIN_CFG['optimizer']['weight_decay'])

n_params = sum(p.numel() for p in encoder.parameters())
print(f'encoder params: {n_params/1e6:.2f} M')
print(f'arcface classes: {len(song_id_map)}  sub_centers: {MODEL_CFG["arcface"]["sub_centers"]}')

## 12. Smoke training (1 epoch)

Before launching the full loop, do a single epoch and check the loss actually moves. With a tiny dataset the absolute value is meaningless — what matters is **monotone decrease**.

In [ ]:
BATCH = min(TRAIN_CFG['loop']['batch_size'], max(2, len(train_ds) // 2))
train_loader = make_loader(train_ds, batch_size=BATCH, shuffle=True,
                           num_workers=TRAIN_CFG['loop']['num_workers'])

smoke = train_one_epoch(
    encoder, loss_fn, optimizer, train_loader,
    device=device, use_amp=TRAIN_CFG['loop']['mixed_precision'],
    grad_clip=TRAIN_CFG['loop']['grad_clip'],
    mixup_alpha=TRAIN_CFG['mixup']['alpha'], mixup_p=TRAIN_CFG['mixup']['p'],
    log_every=5,
)
print('smoke epoch mean loss:', smoke['loss'])

## 13. Full training loop

Now the real loop with cosine schedule, periodic retrieval eval on val, and best-checkpoint tracking. Each `eval_every` epochs we embed the val set, split into gallery/query deterministically, and compute mAP@10.

Tweak `EPOCHS` based on dataset size — with ~50 takes the loss converges in a few epochs; with a real dataset (hundreds of songs, thousands of takes) plan for 100+ epochs.

In [ ]:
EPOCHS = 30                                       # bump up once dataset grows
EVAL_EVERY = max(1, TRAIN_CFG['loop']['eval_every'])
CKPT_DIR = ARTIFACTS_ROOT / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)

steps_per_epoch = max(1, len(train_loader))
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(total_steps * TRAIN_CFG['schedule']['warmup_ratio'])

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + np.cos(np.pi * progress))

scheduler = LambdaLR(optimizer, lr_lambda)
state = TrainState()
history = []

for epoch in range(1, EPOCHS + 1):
    print(f'\n=== epoch {epoch}/{EPOCHS} ===')
    metrics = train_one_epoch(
        encoder, loss_fn, optimizer, train_loader,
        device=device, scheduler=scheduler,
        use_amp=TRAIN_CFG['loop']['mixed_precision'],
        grad_clip=TRAIN_CFG['loop']['grad_clip'],
        mixup_alpha=TRAIN_CFG['mixup']['alpha'], mixup_p=TRAIN_CFG['mixup']['p'],
        log_every=20,
    )
    state.epoch = epoch
    record = {'epoch': epoch, 'train_loss': metrics['loss']}

    if epoch % EVAL_EVERY == 0 and len(val_ds) >= 2:
        eval_out = evaluate_retrieval(encoder, val_ds,
                                      is_gallery_fn=lambda tid: is_gallery_take(tid, seed=SPLIT_SEED),
                                      device=device, batch_size=32, k_for_map=10)
        overall = eval_out['overall']
        record.update({f'val_{k}': overall[k] for k in ('top1', 'top5', 'mrr', 'map_at_10')})
        record['val_n_query'] = eval_out['n_query']
        record['val_n_gallery'] = eval_out['n_gallery']
        print(f'  val: top1={overall["top1"]:.3f}  top5={overall["top5"]:.3f}  '
              f'mrr={overall["mrr"]:.3f}  mAP@10={overall["map_at_10"]:.3f}  '
              f'(N_q={eval_out["n_query"]}, N_g={eval_out["n_gallery"]})')
        if overall['map_at_10'] > state.best_map:
            state.best_map = overall['map_at_10']
            save_checkpoint(CKPT_DIR / 'best_map.pt',
                            model=encoder, loss_fn=loss_fn,
                            optimizer=optimizer, scheduler=scheduler, state=state)
            print(f'  ★ new best mAP@10 — saved checkpoint')

    history.append(record)
    save_checkpoint(CKPT_DIR / 'last.pt',
                    model=encoder, loss_fn=loss_fn,
                    optimizer=optimizer, scheduler=scheduler, state=state)

print('\ntraining done. best val mAP@10:', state.best_map)

## 14. Training curves

In [ ]:
import pandas as pd

df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df['epoch'], df['train_loss'], marker='o')
axes[0].set_title('train loss'); axes[0].set_xlabel('epoch'); axes[0].grid(True, alpha=0.3)
if 'val_map_at_10' in df.columns:
    eval_df = df.dropna(subset=['val_map_at_10'])
    axes[1].plot(eval_df['epoch'], eval_df['val_map_at_10'], marker='o', label='mAP@10')
    axes[1].plot(eval_df['epoch'], eval_df['val_top1'], marker='s', label='top-1')
    axes[1].plot(eval_df['epoch'], eval_df['val_top5'], marker='^', label='top-5')
    axes[1].set_title('val retrieval'); axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 15. Load the best checkpoint and build a FAISS gallery

For evaluation and inference we always use the **best** checkpoint, not the last.

In [ ]:
best_path = CKPT_DIR / 'best_map.pt'
if best_path.exists():
    _ = load_checkpoint(best_path, model=encoder, loss_fn=loss_fn)
    print('loaded', best_path)
else:
    print('no best checkpoint — using current weights')

id_to_slug = {v: k for k, v in val_id_map.items()}
val_gallery = build_gallery(encoder, val_ds, id_to_slug=id_to_slug, device=device, batch_size=32)
GALLERY_PATH = ARTIFACTS_ROOT / 'val_gallery.npz'
val_gallery.save(GALLERY_PATH)
print(f'gallery: {val_gallery.embeddings.shape}  saved to {GALLERY_PATH}')

## 16. Final retrieval evaluation (per-style breakdown)

PLAN.md [12] requires reporting by **style** (cantar / cantarolar / assobiar) — that's the diagnostic that tells us where the model is weakest.

In [ ]:
final = evaluate_retrieval(encoder, val_ds,
                           is_gallery_fn=lambda tid: is_gallery_take(tid, seed=SPLIT_SEED),
                           device=device, batch_size=32, k_for_map=10)
print('OVERALL:', final['overall'])
print()
for style, m in final['per_style'].items():
    print(f'{style:<12}  top1={m["top1"]:.3f}  top5={m["top5"]:.3f}  mAP@10={m["map_at_10"]:.3f}  (N={m["n_queries"]})')

(ARTIFACTS_ROOT / 'eval_val.json').write_text(json.dumps(final, indent=2))

## 17. Inference on a single WAV

Pick any take and run the full sliding-window inference. The result is a top-K of song slugs scored by sum-of-top-3 cosine similarity across windows.

Note: a take from `val_rows` should match itself in the gallery — useful as a sanity check, but real inference would be over an unseen recording.

In [ ]:
mel_mean = torch.tensor(stats['mean'], dtype=torch.float32)
mel_std  = torch.tensor(stats['std'],  dtype=torch.float32)

demo_path = val_rows[0].audio_path if val_rows else train_rows[0].audio_path
result = infer_wav(
    demo_path,
    model=encoder, gallery=val_gallery,
    mel_cfg=mel_cfg, pesto_cfg=pesto_cfg,
    mel_mean=mel_mean, mel_std=mel_std,
    window_s=INFER_CFG['window_s'], hop_s=INFER_CFG['hop_s'],
    top_k_per_window=INFER_CFG['top_k_per_window'],
    top_n_return=INFER_CFG['top_n_return'],
    device=device,
    target_lufs=PREPROC['loudness']['target_lufs'],
    trim_step_size_ms=PREPROC['trim']['step_size_ms'],
    trim_confidence_threshold=PREPROC['trim']['confidence_threshold'],
    trim_margin_ms=PREPROC['trim']['margin_ms'],
)
print(f'WAV: {demo_path.name}  windows={result.n_windows}')
for rank, (slug, score) in enumerate(result.top, 1):
    print(f'  {rank}. {slug:<30} score={score:.3f}')

## 18. Save everything to Drive

By now Drive already has `splits.json`, `stats.json`, `checkpoints/best_map.pt`, `checkpoints/last.pt`, `val_gallery.npz`, `eval_val.json`. We also dump the resolved configs and training history so the run is reproducible.

In [ ]:
(ARTIFACTS_ROOT / 'configs_resolved.json').write_text(json.dumps({
    'preproc': PREPROC, 'model': MODEL_CFG, 'train': TRAIN_CFG, 'infer': INFER_CFG,
    'split_seed': SPLIT_SEED, 'ratios': list(RATIOS), 'epochs_run': len(history),
    'song_id_map_size': len(song_id_map),
}, indent=2))
(ARTIFACTS_ROOT / 'history.json').write_text(json.dumps(history, indent=2))

import os
for p in sorted(ARTIFACTS_ROOT.rglob('*')):
    if p.is_file():
        print(f'{p.relative_to(ARTIFACTS_ROOT)}  ({os.path.getsize(p)/1024:.1f} KB)')

---

## Next steps

- **Grow the dataset.** Per PLAN.md, ArcFace generalizes only when `N_train ≫ N_catalogue` (≈5×). Until then, all metrics are noisy.
- **Calibrate the rejection threshold τ** on val (PLAN.md [11.3]) once enough negatives exist — use queries from songs *not* in the val gallery as the negative pool.
- **Run on test set** with frozen splits/encoder for the final reportable numbers.
- **Cross-style breakdown**: query in `cantar` vs gallery restricted to `assobiar` etc. (PLAN.md [12]). The current `evaluate_retrieval` already breaks down by query style; restricting the gallery is a small extension.
- **Move to a real GPU box** when training time exceeds a Colab session (12 h). Re-use `scripts/` from the repo — they import the same `src/` modules.